### Representación ampliada

| Variable                  | Qué representa                                               |
| ------------------------- | ------------------------------------------------------------ |
| `days_since_previous_order` | días desde el pedido anterior al último pedido histórico     |
| `frequency`               | número de pedidos históricos                                 |
| `avg_basket_size`         | tamaño medio de cesta                                        |
| `std_basket_size`         | variabilidad del tamaño de cesta                             |
| `avg_days_between_orders` | tiempo medio entre pedidos                                   |
| `std_days_between_orders` | regularidad de compra                                        |
| `unique_products`         | variedad de productos comprados                              |
| `unique_aisles`           | variedad de pasillos/categorías                              |
| `unique_departments`      | variedad de departamentos                                    |
| `reorder_ratio`           | proporción de productos que son recompras                    |
| `product_variety_ratio`   | proporción de productos distintos respecto al total comprado |


In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../../data/instacart")
DATA_DIR_CLEAN = Path("../../data_without_nulls/instacart")

In [2]:
instacart_extended = pd.read_csv("../../simple_datasets/instacart_model_simple.csv")


orders = pd.read_csv(DATA_DIR_CLEAN / "orders.csv")
order_products_prior = pd.read_csv(DATA_DIR / "order_products__prior.csv")
aisles = pd.read_csv(DATA_DIR / "aisles.csv")
departments = pd.read_csv(DATA_DIR / "departments.csv")
products = pd.read_csv(DATA_DIR / "products.csv")

orders_prior = orders[
    orders["eval_set"] == "prior"
].copy()



### Variabilidad de tamaño de cesta

In [3]:
# Número de productos de cada pedido
basket_per_order = (
    order_products_prior
    .groupby("order_id")
    .size()
    .reset_index(name="basket_size")
)

# Asociamos cada pedido con su usuario
basket_per_order = basket_per_order.merge(
    orders_prior[["order_id", "user_id"]],
    on="order_id",
    how="left"
)

# Ya tenemos avg_basket_size en el dataset simple,
# por lo que ahora solamente necesitamos su desviación
basket_variability = (
    basket_per_order
    .groupby("user_id")["basket_size"]
    .std()
    .reset_index(name="std_basket_size")
)

basket_variability["std_basket_size"] = (
    basket_variability["std_basket_size"].fillna(0)
)

### Comportamiento temporal

In [4]:
# Utilizamos todos los intervalos entre pedidos disponibles
temporal_orders = orders[
    orders["days_since_prior_order"].notna()
][["user_id", "days_since_prior_order"]]

temporal_features = (
    temporal_orders
    .groupby("user_id")["days_since_prior_order"]
    .agg(
        avg_days_between_orders="mean",
        std_days_between_orders="std"
    )
    .reset_index()
)

temporal_features["std_days_between_orders"] = (
    temporal_features["std_days_between_orders"].fillna(0)
)

### Información de productos

In [5]:
# Añadimos aisle y department a cada producto comprado
purchases = order_products_prior.merge(
    products[
        ["product_id", "aisle_id", "department_id"]
    ],
    on="product_id",
    how="left"
)

# Añadimos el usuario correspondiente a cada pedido
purchases = purchases.merge(
    orders_prior[
        ["order_id", "user_id"]
    ],
    on="order_id",
    how="left"
)

### Número de productos diferentes

In [6]:
unique_products = (
    purchases
    .groupby("user_id")["product_id"]
    .nunique()
    .reset_index(name="unique_products")
)

### Número de aisles diferentes

In [7]:
unique_aisles = (
    purchases
    .groupby("user_id")["aisle_id"]
    .nunique()
    .reset_index(name="unique_aisles")
)

### Número de departamentos diferentes

In [8]:
unique_departments = (
    purchases
    .groupby("user_id")["department_id"]
    .nunique()
    .reset_index(name="unique_departments")
)


### Proporción de recompras

In [9]:
reorder_ratio = (
    purchases
    .groupby("user_id")["reordered"]
    .mean()
    .reset_index(name="reorder_ratio")
)


### Variedad de productos

In [10]:
total_products = (
    purchases
    .groupby("user_id")
    .size()
    .reset_index(name="total_products")
)

# Unimos con el número de productos distintos
product_variety = total_products.merge(
    unique_products,
    on="user_id",
    how="left"
)

# Proporción de productos distintos respecto al total comprado
product_variety["product_variety_ratio"] = (
    product_variety["unique_products"]
    / product_variety["total_products"]
)

product_variety = product_variety[
    ["user_id", "product_variety_ratio"]
]


### Adición de nuevas variables al dataset

In [11]:
instacart_extended = (
    instacart_extended
    .merge(basket_variability, on="user_id", how="left")
    .merge(temporal_features, on="user_id", how="left")
    .merge(unique_products, on="user_id", how="left")
    .merge(unique_aisles, on="user_id", how="left")
    .merge(unique_departments, on="user_id", how="left")
    .merge(reorder_ratio, on="user_id", how="left")
    .merge(product_variety, on="user_id", how="left")
)


In [12]:
instacart_extended.head()

,user_id,days_since_previous_order,frequency,avg_basket_size,std_basket_size,avg_days_between_orders,std_days_between_orders,unique_products,unique_aisles,unique_departments,reorder_ratio,product_variety_ratio
0,1,14.0,10,5.900000,1.523884,17.272727,10.306221,18,12,7,0.694915,0.305085
1,2,30.0,14,13.928571,5.717238,15.200000,10.751744,102,33,13,0.476923,0.523077
2,3,11.0,12,7.333333,2.103388,11.076923,5.936631,33,16,9,0.625000,0.375000
3,4,30.0,5,3.600000,2.073644,14.166667,12.023588,17,14,9,0.055556,0.944444
4,5,6.0,4,9.250000,3.095696,9.200000,6.978539,23,16,9,0.378378,0.621622


In [13]:
print("Dimensiones:", instacart_extended.shape)

print("\nColumnas:")
print(instacart_extended.columns.tolist())

print("\nUsuarios duplicados:")
print(instacart_extended["user_id"].duplicated().sum())

print("\nValores nulos:")
print(instacart_extended.isnull().sum())

print("\nEstadísticas:")
display(
    instacart_extended
    .drop(columns="user_id")
    .describe()
    .T
)


Dimensiones: (206209, 12)

Columnas:
['user_id', 'days_since_previous_order', 'frequency', 'avg_basket_size', 'std_basket_size', 'avg_days_between_orders', 'std_days_between_orders', 'unique_products', 'unique_aisles', 'unique_departments', 'reorder_ratio', 'product_variety_ratio']

Usuarios duplicados:
0

Valores nulos:
user_id                    0
days_since_previous_order                    0
frequency                  0
avg_basket_size            0
std_basket_size            0
avg_days_between_orders    0
std_days_between_orders    0
unique_products            0
unique_aisles              0
unique_departments         0
reorder_ratio              0
product_variety_ratio      0
dtype: int64

Estadísticas:


,count,mean,std,min,25%,50%,75%,max
days_since_previous_order,206209.0,17.061782,10.672178,0.000000,7.000000,15.000000,30.000000,30.000000
frequency,206209.0,15.590367,16.654774,3.000000,5.000000,9.000000,19.000000,99.000000
avg_basket_size,206209.0,9.951586,5.863570,1.000000,5.740741,8.933333,13.000000,70.250000
std_basket_size,206209.0,4.266349,2.675061,0.000000,2.345208,3.781534,5.609516,44.747439
avg_days_between_orders,206209.0,13.428600,5.496185,0.000000,9.000000,13.320000,17.687500,27.000000
std_days_between_orders,206209.0,9.478742,3.429753,0.000000,7.029160,9.976867,12.066734,17.320508
unique_products,206209.0,64.536238,56.592339,1.000000,25.000000,48.000000,86.000000,726.000000
unique_aisles,206209.0,27.783700,16.111530,1.000000,15.000000,25.000000,38.000000,102.000000
unique_departments,206209.0,10.827796,3.791768,1.000000,8.000000,11.000000,14.000000,21.000000
reorder_ratio,206209.0,0.432249,0.212144,0.000000,0.267857,0.428571,0.595745,0.989529


### Guardado del dataset ampliado

In [ ]:
instacart_extended.to_csv(
    "../../extended_datasets/instacart_model_extended.csv",
    index=False
)